# Veritas Bureau — Notebook 1: Pipeline Demonstration
**IE University · Gen AI & Agents Course · April 2026**

This notebook demonstrates the multi-agent fake news detection pipeline built with plain Python + Ollama (no frameworks like CrewAI).

---

## 1. Architecture Overview

```
Article Text
     │
     ▼
┌─────────────────┐
│  Agent 1        │  Claim Extractor — extracts 3-5 verifiable factual claims
│  (sequential)   │
└────────┬────────┘
         │
         ▼
┌────────────────────────────────────────────┐
│  Parallel Execution (ThreadPoolExecutor)   │
│  ┌──────────┐ ┌──────────┐ ┌────────────┐ │
│  │ Agent 2  │ │ Agent 3  │ │  Agent 5   │ │
│  │ Fact     │ │ Bias     │ │  Source    │ │
│  │ Checker  │ │ Detector │ │  Credib.   │ │
│  └──────────┘ └──────────┘ └────────────┘ │
└─────────────────┬──────────────────────────┘
                  │
                  ▼
         ┌────────────────┐
         │   Agent 4      │  Judge — synthesizes all evidence → verdict
         │   (sequential) │
         └───────┬────────┘
                 │
         confidence < 0.6?
                 │ YES
                 ▼
         ┌────────────────┐
         │   Agent 6      │  Devil's Advocate — stress-tests the verdict
         │   (conditional)│
         └───────┬────────┘
                 │
                 ▼
           REAL / FAKE
```

**Why no CrewAI?** Plain Python with `ThreadPoolExecutor` for parallel agents gives ~7x less overhead than CrewAI's agent orchestration framework. Each agent is just a function calling `ollama.chat()` directly.

## 2. Setup

In [1]:
import sys, os
sys.path.insert(0, os.path.dirname(os.getcwd()))  # add project root to path

# If running from notebooks/ dir, also try parent
if not os.path.exists('config.py'):
    os.chdir('..')

print(f"Working directory: {os.getcwd()}")
print("Imports ready")

Working directory: <PROJECT_DIR>/fake_news_mas_v1
Imports ready


In [2]:
from config import BACKEND, OLLAMA_MODEL, call_llm
from agents.prompts import (
    CLAIM_EXTRACTOR_PROMPT, FACT_CHECKER_PROMPT, BIAS_DETECTOR_PROMPT,
    SOURCE_CREDIBILITY_PROMPT, JUDGE_PROMPT, DEVILS_ADVOCATE_PROMPT
)
from agents.runners import (
    run_claim_extractor, run_fact_checker, run_bias_detector,
    run_source_credibility, run_judge, run_devils_advocate
)
from pipeline.core import analyze_article
from pipeline.json_parser import extract_json
import json

print(f"Backend: {BACKEND}")
print(f"Model: {OLLAMA_MODEL}")

Backend: ollama
Model: qwen2.5:14b


## 3. Agent Prompts

Each agent has a dedicated system prompt stored in `agents/prompts.py`.

In [3]:
prompts = {
    "Agent 1 — Claim Extractor": CLAIM_EXTRACTOR_PROMPT,
    "Agent 2 — Fact Checker": FACT_CHECKER_PROMPT,
    "Agent 3 — Bias Detector": BIAS_DETECTOR_PROMPT,
    "Agent 5 — Source Credibility": SOURCE_CREDIBILITY_PROMPT,
    "Agent 4 — Judge": JUDGE_PROMPT,
    "Agent 6 — Devil's Advocate": DEVILS_ADVOCATE_PROMPT,
}

for name, prompt in prompts.items():
    print(f"{'='*60}")
    print(f"  {name}")
    print(f"{'='*60}")
    print(prompt[:400] + ('...' if len(prompt) > 400 else ''))
    print()

  Agent 1 — Claim Extractor
You are a veteran investigative journalist with 20 years of experience at Reuters and the Associated Press. You have an exceptional eye for distinguishing factual claims from opinion, rhetoric, and editorialization.

Your task: Read the provided news article and extract the requested number of key factual claims that can be verified. Focus on statistics, named entities, events, dates, and specific...

  Agent 2 — Fact Checker
You are a senior researcher at a fact-checking organization similar to Snopes or PolitiFact. You have deep knowledge of world events, politics, science, and economics.

Your task: For each claim, you are given live web search snippets retrieved at analysis time. Use them as your primary evidence, supplementing with your own knowledge where snippets are absent or insufficient. Determine if each cla...

  Agent 3 — Bias Detector
You are a linguistics professor specializing in media studies and computational rhetoric at Columbia Journalism

## 4. Article Walkthroughs

We run three articles through the full pipeline: one clearly **FAKE**, one clearly **REAL**, and one **AMBIGUOUS**.

In [4]:
ARTICLES = [
    {
        "expected": "FAKE",
        "title": "BREAKING: Hillary Clinton Arrested for Treason — Deep State Exposed",
        "text": (
            "Sources close to the White House confirmed Thursday that Hillary Clinton was secretly arrested "
            "by military tribunals for treason against the United States. The Deep State operation, which "
            "has been running for decades, was finally exposed by patriot insiders. Clinton was allegedly "
            "caught selling classified information to China and funding radical terrorist groups. President "
            "Trump reportedly signed the arrest warrant last night. Mainstream media refuses to cover this "
            "EXPLOSIVE story because they are complicit. Share this before it gets deleted! The arrests "
            "are just beginning — hundreds of globalist elites will face military justice."
        ),
    },
    {
        "expected": "REAL",
        "title": "Federal Reserve raises interest rates by 0.25 percentage points",
        "text": (
            "The Federal Reserve on Wednesday raised its benchmark interest rate by a quarter of a percentage "
            "point, as policymakers continued their effort to bring inflation back down to the central bank's "
            "2 percent target. The decision was unanimous among the 12 voting members of the Federal Open "
            "Market Committee. Fed Chair Jerome Powell said in a press conference that while inflation has "
            "eased somewhat, it remains above the Fed's longer-run goal. The new target range for the "
            "federal funds rate is now 5.25 to 5.5 percent, the highest level in 22 years."
        ),
    },
    {
        "expected": "AMBIGUOUS",
        "title": "Scientists discover ancient virus frozen in Siberian permafrost",
        "text": (
            "French scientists have revived a 48,500-year-old virus from Siberian permafrost, raising concerns "
            "about potential pandemic risks as Arctic ice continues to melt. The virus, called Pandoravirus yedoma, "
            "was extracted from a frozen lake sediment sample in Russia. While the researchers say the virus can "
            "only infect amoeba and poses no direct risk to humans, some biosafety experts warn that the melting "
            "permafrost could release unknown ancient pathogens. The study was published in the journal Viruses."
        ),
    },
]
print(f"Loaded {len(ARTICLES)} test articles")

Loaded 3 test articles


In [5]:
# Run all 3 articles through the pipeline
results = []
for article in ARTICLES:
    print(f"\n{'='*60}")
    print(f"[{article['expected']}] {article['title']}")
    print(f"{'='*60}")
    result = analyze_article(article["text"], article["title"])
    result["expected"] = article["expected"]
    results.append(result)
    print(f"→ Verdict: {result['final_label']} ({result['final_confidence']:.1%}) in {result['total_processing_time']}s")


[FAKE] BREAKING: Hillary Clinton Arrested for Treason — Deep State Exposed
[Pipeline] Running Agent 1: Claim Extractor...
[Pipeline] Running Agents 2+3+5+HF in parallel (Fact Check, Bias, Credibility, HF Classifier)...
[Fact Checker] Searching web for 1 claim(s)...


/opt/anaconda3/envs/hw2_env311/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


[Fact Checker] Retrieved 3 web snippet(s).
[Pipeline] HF Classifier: FAKE (100%)
[Pipeline] Running Agent 4: Judge...
[Pipeline] Complete in 93.76s. Verdict: FAKE (0.98)
→ Verdict: FAKE (98.0%) in 93.76s

[REAL] Federal Reserve raises interest rates by 0.25 percentage points
[Pipeline] Running Agent 1: Claim Extractor...
[Pipeline] Running Agents 2+3+5+HF in parallel (Fact Check, Bias, Credibility, HF Classifier)...
[Fact Checker] Searching web for 3 claim(s)...
[Fact Checker] Retrieved 9 web snippet(s).
[Pipeline] HF Classifier: REAL (100%)
[Pipeline] Running Agent 4: Judge...
[Pipeline] Low confidence (0.75) -> triggering Agent 6: Devil's Advocate...
[Pipeline] Verdict UPHELD. Revised confidence: 0.65
[Pipeline] Complete in 190.88s. Verdict: REAL (0.65)
→ Verdict: REAL (65.0%) in 190.88s

[AMBIGUOUS] Scientists discover ancient virus frozen in Siberian permafrost
[Pipeline] Running Agent 1: Claim Extractor...
[Pipeline] Running Agents 2+3+5+HF in parallel (Fact Check, Bias, Credibili

In [6]:
# Detailed display for each result
for result in results:
    print(f"\n{'#'*60}")
    print(f"# {result['title']}")
    print(f"# Expected: {result['expected']}  |  Got: {result['final_label']} ({result['final_confidence']:.1%})")
    print(f"{'#'*60}")
    
    print("\n--- CLAIMS EXTRACTED ---")
    for c in result["claims"].get("claims", []):
        print(f"  • {c}")
    
    print("\n--- FACT CHECK ---")
    for r in result["fact_check"].get("results", []):
        print(f"  [{r.get('verdict')}] {r.get('claim','')[:70]}")
        print(f"         ↳ {r.get('evidence','')}")
    
    bias = result["bias"]
    print(f"\n--- BIAS ---")
    print(f"  Tone: {bias.get('tone')}  |  Score: {bias.get('bias_score')}")
    for f in bias.get("flags", []):
        print(f"  ⚑ {f}")
    
    cred = result["credibility"]
    print(f"\n--- SOURCE CREDIBILITY ---")
    print(f"  Score: {cred.get('credibility_score')}  |  Headline: {cred.get('headline_quality')}")
    for f in cred.get("flags", []):
        print(f"  ⚑ {f}")
    
    print(f"\n--- JUDGE ---")
    print(f"  Summary: {result['judge'].get('summary','')}")
    
    if result["devils_advocate_triggered"]:
        da = result["devils_advocate"]
        print(f"\n--- DEVIL'S ADVOCATE ---")
        print(f"  Recommendation: {da.get('recommendation')}")
        print(f"  Reasoning: {da.get('reasoning','')}")
    
    print(f"\n  Total time: {result['total_processing_time']}s")


############################################################
# BREAKING: Hillary Clinton Arrested for Treason — Deep State Exposed
# Expected: FAKE  |  Got: FAKE (98.0%)
############################################################

--- CLAIMS EXTRACTED ---

--- FACT CHECK ---
  [CONTRADICTED] No specific claims could be extracted from this article.
         ↳ The original article context clearly outlines several specific claims, including Hillary Clinton's arrest for treason and her involvement with China and terrorist groups.

--- BIAS ---
  Tone: inflammatory  |  Score: 1.0
  ⚑ Emotionally loaded headline
  ⚑ No counter-arguments presented
  ⚑ Unattributed quotes
  ⚑ Fear-mongering or urgency language

--- SOURCE CREDIBILITY ---
  Score: 0.15  |  Headline: sensational
  ⚑ No visible sourcing or attribution
  ⚑ Headline is highly sensational and clickbait-like
  ⚑ Article relies on unsupported assertions and lacks evidence

--- JUDGE ---
  Summary: The article makes a highly specific

## 5. Performance: Why Not CrewAI?

CrewAI introduces significant overhead per agent invocation (task routing, event loops, agent memory serialization). Our approach:

| Approach | Per-article overhead | Parallel support |
|---|---|---|
| CrewAI framework | ~2–5s framework overhead | Sequential by default |
| Plain Python + Ollama | ~0.05s | Native ThreadPoolExecutor |

With 3 agents running in parallel (Agents 2+3+5), our effective pipeline time is:
`T_total ≈ T_agent1 + max(T_agent2, T_agent3, T_agent5) + T_agent4`

Rather than `T_total ≈ T1 + T2 + T3 + T4 + T5` (sequential).

In [7]:
# Show per-agent timing breakdown
for result in results:
    print(f"\n{result['title'][:55]}...")
    agents = [
        ("Claim Extractor", result["claims"]),
        ("Fact Checker", result["fact_check"]),
        ("Bias Detector", result["bias"]),
        ("Source Credibility", result["credibility"]),
        ("Judge", result["judge"]),
    ]
    if result["devils_advocate"]:
        agents.append(("Devil's Advocate", result["devils_advocate"]))
    
    for name, data in agents:
        t = data.get("_processing_time", 0)
        bar = "█" * int(t * 2)
        print(f"  {name:<22} {t:5.1f}s  {bar}")
    print(f"  {'TOTAL':<22} {result['total_processing_time']:5.1f}s")


BREAKING: Hillary Clinton Arrested for Treason — Deep S...
  Claim Extractor          5.3s  ██████████
  Fact Checker            49.5s  ██████████████████████████████████████████████████████████████████████████████████████████████████
  Bias Detector           41.2s  ██████████████████████████████████████████████████████████████████████████████████
  Source Credibility      49.8s  ███████████████████████████████████████████████████████████████████████████████████████████████████
  Judge                   38.6s  █████████████████████████████████████████████████████████████████████████████
  TOTAL                   93.8s

Federal Reserve raises interest rates by 0.25 percentag...
  Claim Extractor         15.7s  ███████████████████████████████
  Fact Checker            98.3s  ████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████
  Bias Detector  

## 6. JSON Parser — Robustness Demo

In [8]:
# Demonstrate the JSON parser handling messy LLM output
from pipeline.json_parser import extract_json

messy_inputs = [
    # Markdown fences
    '```json\n{"label": "FAKE", "confidence": 0.9}\n```',
    # Leading text
    'Here is my analysis:\n{"label": "REAL", "confidence": 0.7}',
    # Trailing text
    '{"label": "FAKE", "confidence": 0.8} I hope this helps!',
    # Clean JSON
    '{"label": "REAL", "confidence": 0.95}',
]

default = {"label": "UNKNOWN", "confidence": 0.5}
for inp in messy_inputs:
    parsed = extract_json(inp, default=default)
    print(f"Input:  {inp[:60]!r}")
    print(f"Parsed: {parsed}")
    print()

Input:  '```json\n{"label": "FAKE", "confidence": 0.9}\n```'
Parsed: {'label': 'FAKE', 'confidence': 0.9}

Input:  'Here is my analysis:\n{"label": "REAL", "confidence": 0.7}'
Parsed: {'label': 'REAL', 'confidence': 0.7}

Input:  '{"label": "FAKE", "confidence": 0.8} I hope this helps!'
Parsed: {'label': 'FAKE', 'confidence': 0.8}

Input:  '{"label": "REAL", "confidence": 0.95}'
Parsed: {'label': 'REAL', 'confidence': 0.95}

